# 04. SatChecker Query for DECam Streak
Written by Kiyoaki Okudaira and Meredith Rawls<br>
*University of Washington / IAU CPS SatHub<br>
(kiyoaki@uw.edu or okudaira.kiyoaki.528@s.kyushu-u.ac.jp)<br>
<br>
This code is written for ASTR 499 undergraduate research with Dr. Meredith.<br>
A notebook to query the SatChecker Field Of View (FOV) service.<br>
<br>
**History**<br>
coding 2026-02-21 : 1st coding

### Import and initial settings
**Standard libraries**

In [1]:
from os import path
import pickle
import json
from tqdm.notebook import tqdm

import requests

import astropy.units as u
from astropy.time import Time
from astropy.coordinates import EarthLocation
from astropy.io import fits
from astropy.coordinates import SkyCoord

from concurrent.futures import ThreadPoolExecutor, as_completed

**Import file settings**

In [ ]:
# project directory
PATH_project = '/astro/store/shire/kiyoaki/ASTR499/'

PATH_input   = PATH_project + 'input/'
PATH_output  = PATH_project + 'output/'

PATH_image   = PATH_input  + 'fits_data/'
PATH_session = PATH_output + 'session/'

# ORIGINAL streak list from Alex
fname_streak_list = 'streaks_augmented_20230817.csv'
PATH_streak_list  = fname_streak_list + 'decam_streak_list/' + fname_streak_list

**Process method setting**

In [ ]:
paralell_process = True
if paralell_process:
    from concurrent.futures import ThreadPoolExecutor, as_completed

### Load Streak Dataset
Dataset is grouped by EXPNUM because if the EXPNUM is the same, the request to satchecker is the same．

In [ ]:
with open(PATH_session+path.splitext(path.basename(fname_streak_list))[0]+'_02_masked_by_gauss.pkl', 'rb') as f:
    streak_table = pickle.load(f)
streak_table = streak_table.group_by("expnum")
output = []

### Hard-wired DECam parameters

Original source: https://github.com/DarkEnergySurvey/drawDECam

In [4]:
CCDSECTIONS = {
    1: [2049, 4096, 8193, 12288],
    2: [2049, 4096, 12289, 16384],
    3: [2049, 4096, 16385, 20480],
    4: [4097, 6144, 6145, 10240],
    5: [4097, 6144, 10241, 14336],
    6: [4097, 6144, 14337, 18432],
    7: [4097, 6144, 18433, 22528],
    8: [6145, 8192, 4097, 8192],
    9: [6145, 8192, 8193, 12288],
    10: [6145, 8192, 12289, 16384],
    11: [6145, 8192, 16385, 20480],
    12: [6145, 8192, 20481, 24576],
    13: [8193, 10240, 2049, 6144],
    14: [8193, 10240, 6145, 10240],
    15: [8193, 10240, 10241, 14336],
    16: [8193, 10240, 14337, 18432],
    17: [8193, 10240, 18433, 22528],
    18: [8193, 10240, 22529, 26624],
    19: [10241, 12288, 2049, 6144],
    20: [10241, 12288, 6145, 10240],
    21: [10241, 12288, 10241, 14336],
    22: [10241, 12288, 14337, 18432],
    23: [10241, 12288, 18433, 22528],
    24: [10241, 12288, 22529, 26624],
    25: [12289, 14336, 1, 4096],
    26: [12289, 14336, 4097, 8192],
    27: [12289, 14336, 8193, 12288],
    28: [12289, 14336, 12289, 16384],
    29: [12289, 14336, 16385, 20480],
    30: [12289, 14336, 20481, 24576],
    31: [12289, 14336, 24577, 28672],
    32: [14337, 16384, 1, 4096],
    33: [14337, 16384, 4097, 8192],
    34: [14337, 16384, 8193, 12288],
    35: [14337, 16384, 12289, 16384],
    36: [14337, 16384, 16385, 20480],
    37: [14337, 16384, 20481, 24576],
    38: [14337, 16384, 24577, 28672],
    39: [16385, 18432, 2049, 6144],
    40: [16385, 18432, 6145, 10240],
    41: [16385, 18432, 10240, 14335],
    42: [16385, 18432, 14336, 18431],
    43: [16385, 18432, 18432, 22527],
    44: [16385, 18432, 22528, 26623],
    45: [18433, 20480, 2049, 6144],
    46: [18433, 20480, 6145, 10240],
    47: [18433, 20480, 10240, 14335],
    48: [18433, 20480, 14336, 18431],
    49: [18433, 20480, 18432, 22527],
    50: [18433, 20480, 22528, 26623],
    51: [20481, 22528, 4097, 8192],
    52: [20481, 22528, 8193, 12288],
    53: [20481, 22528, 12289, 16384],
    54: [20481, 22528, 16385, 20480],
    55: [20481, 22528, 20481, 24576],
    56: [22529, 24576, 6145, 10240],
    57: [22529, 24576, 10240, 14335],
    58: [22529, 24576, 14336, 18431],
    59: [22529, 24576, 18432, 22527],
    60: [24577, 26624, 8193, 12288],
    61: [24577, 26624, 12289, 16384],
    62: [24577, 26624, 16385, 20480]
}

CCDSECTION_X0 = (CCDSECTIONS[28][1] + CCDSECTIONS[35][0]) / 2.0
CCDSECTION_Y0 = (CCDSECTIONS[35][2] + CCDSECTIONS[28][3]) / 2.0

TRIM_CCDSECTIONS = CCDSECTIONS.copy()
borderpix = 104  # 208/2. as 208 is the space between chips in pixels
for _k, _v in list(TRIM_CCDSECTIONS.items()):
    (_x1, _x2, _y1, _y2) = _v
    _x1 = _x1 + borderpix
    _x2 = _x2 - borderpix
    _y1 = _y1 + borderpix
    _y2 = _y2 - borderpix
    TRIM_CCDSECTIONS[_k] = [_x1, _x2, _y1, _y2]

def createDECam_TANheader(ra_center, dec_center, pixscale=0.2634):
    """
    Creates a fake TAN projection header for DECam image to project the
    CCD Sections on the sky
    pixscale is in arcseconds per pixel
    """
    DECam_header = {
        'CTYPE1': 'RA---TAN',  # / WCS projection type for this axis
        'CTYPE2': 'DEC--TAN',  # / WCS projection type for this axis
        'CUNIT1': 'deg',  # / Axis unit
        'CUNIT2': 'deg',  # / Axis unit
        'CRVAL1': ra_center,  # / World coordinate on this axis
        'CRPIX1': CCDSECTION_X0,  # / Reference pixel on this axis
        'CD1_1': 0,  # /
        'CD1_2': +pixscale / 3600.,  # /
        'CRVAL2': dec_center,  # /
        'CRPIX2': CCDSECTION_Y0,  # /
        'CD2_1': -pixscale / 3600.,  # /
        'CD2_2': 0.  # /
    }
    return DECam_header

### SatChecker query parameters

Must specify the observatory location, the sky region, and the time and duration of the observation.<br>
Docs available at https://satchecker.readthedocs.io/en/latest/fov.html<br>
<br>
**FOV and Telescope location**

In [5]:
fov_radius = 1.5  # degree radius for the satchecker query
# DECam FOV is 3 square degrees (2.2 degrees across)
location = EarthLocation.of_site('ctio')
latitude = location.lat.value  # deg
longitude = location.lon.value  # deg
elevation = location.height.value  # meters

**Satchecker query (single processing)**

In [ ]:
if paralell_process is False:
    for group in tqdm(streak_table.groups):
        basename = path.basename(group[0]["archive_filename"])
        md5sum = group[0]["md5sum"]
        expnum = group[0]["expnum"]
        ccdnum = group[0]["ccdnum"]
        streak_ID = group[0]["streakID"]

        save_path = PATH_image + path.splitext(path.splitext(basename)[0])[0] + "_CCD_{0}".format(ccdnum) + path.splitext(path.splitext(basename)[0])[1] + path.splitext(basename)[1]
        satchecker_result_path = PATH_output + "satchecker/" + path.splitext(path.splitext(basename)[0])[0] + "_satchecker.json"

        if path.exists(satchecker_result_path):
            continue

        main_header = fits.open(save_path)[0].header

        exp_time = main_header["EXPTIME"] * u.s
        exp_begin = Time(main_header["DATE-OBS"])

        start_time_jd = exp_begin.jd
        duration = exp_time.value

        # Observation time margin for SatChecker query
        start_time_jd = (exp_begin - exp_time).jd
        duration = (exp_time * 3).value

        # telescope coordinate
        coord = SkyCoord(ra=main_header["TELRA"], dec=main_header["TELDEC"], unit=(u.hour, u.deg))
        ra_center = coord.icrs.ra.value
        dec_center = coord.icrs.dec.value

        # Make the SatChecker API request
        url_string = f"https://satchecker.cps.iau.org/fov/satellite-passes/?latitude={latitude}&longitude={longitude}&elevation={elevation}&start_time_jd={start_time_jd}&duration={duration}&ra={ra_center}&dec={dec_center}&fov_radius={fov_radius}&group_by=satellite&async=False"
        response = requests.get(url_string, timeout=60)
        data = response.json()
        print(f"URL : {url_string}")

        with open(PATH_output + "satchecker/" + path.splitext(path.splitext(basename)[0])[0] + "_satchecker.json" ,'w') as json_output:
            json.dump(data, json_output, ensure_ascii=False, indent=4, sort_keys=True, separators=(',', ': '))

  0%|          | 0/3127 [00:00<?, ?it/s]

**Satchecker query (paralell processing)**

In [ ]:
if paralell_process:
    def process_one_group(group):

        basename = path.basename(group[0]["archive_filename"])
        md5sum = group[0]["md5sum"]
        expnum = group[0]["expnum"]
        ccdnum = group[0]["ccdnum"]
        streak_ID = group[0]["streakID"]

        save_path = (
            PATH_image
            + path.splitext(path.splitext(basename)[0])[0]
            + f"_CCD_{ccdnum}"
            + path.splitext(path.splitext(basename)[0])[1]
            + path.splitext(basename)[1]
        )

        satchecker_result_path = (
            PATH_output
            + "satchecker/"
            + path.splitext(path.splitext(basename)[0])[0]
            + "_satchecker.json"
        )

        if path.exists(satchecker_result_path):
            return "exist"

        try:
            with fits.open(save_path) as hdul:
                main_header = hdul[0].header

            exp_time = main_header["EXPTIME"] * u.s
            exp_begin = Time(main_header["DATE-OBS"])

            # SatChecker query margin
            start_time_jd = (exp_begin - exp_time).jd
            duration = (exp_time * 3).value

            coord = SkyCoord(
                ra=main_header["TELRA"],
                dec=main_header["TELDEC"],
                unit=(u.hour, u.deg),
            )

            ra_center = coord.icrs.ra.value
            dec_center = coord.icrs.dec.value

            url_string = (
                f"https://satchecker.cps.iau.org/fov/satellite-passes/"
                f"?latitude={latitude}"
                f"&longitude={longitude}"
                f"&elevation={elevation}"
                f"&start_time_jd={start_time_jd}"
                f"&duration={duration}"
                f"&ra={ra_center}"
                f"&dec={dec_center}"
                f"&fov_radius={fov_radius}"
                f"&group_by=satellite"
                f"&async=False"
            )

            response = requests.get(url_string, timeout=60)
            data = response.json()

            with open(satchecker_result_path, "w") as json_output:
                json.dump(
                    data,
                    json_output,
                    ensure_ascii=False,
                    indent=4,
                    sort_keys=True,
                    separators=(",", ": "),
                )

            return "done"

        except Exception as e:
            print(f"error: {e}")
            return f"error: {e}"


    # -------------------------------
    # 並列実行
    # -------------------------------

    max_workers = 3   # APIに負荷をかけすぎない値

    results = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_one_group, group)
                for group in streak_table.groups]

        for fut in tqdm(as_completed(futures), total=len(futures)):
            results.append(fut.result())


  0%|          | 0/3127 [00:00<?, ?it/s]